# Grafos interactivos y matriz MGCECDL de variables CHEC

Notebook autosuficiente para generar tres grafos desde `src/data/variables.json`:

1. El grafo completo con 156 nodos.
2. El grafo filtrado con las filas que tengan `SELECCIÓN = 1` en el Excel vigente.
3. El grafo MGCECDL con las variables que sobreviven a `procesar_dataset_completo`, excluyendo `UITI_VANO` por ser el objetivo.

El notebook prioriza el archivo `data/Variables_seleccion.xlsx` del proyecto `chec_impacto`. Las conexiones entre variables seleccionadas se preservan atravesando variables retiradas y usando como peso efectivo el mínimo peso del camino original.

También regenera la matriz dirigida de adyacencia en el orden exacto de `features` y guarda:

- `data/graphs/mgcecdl_adjacency_matrix.npy`
- `data/graphs/mgcecdl_feature_order.json`
- `data/graphs/mgcecdl_preserved_edges.json`

In [1]:
# Ejecutar solo si el entorno no tiene instaladas estas dependencias.
%pip install networkx pyvis openpyxl

Note: you may need to restart the kernel to use updated packages.


In [2]:
import json
import os
import sys
from pathlib import Path

import numpy as np

import networkx as nx
from openpyxl import load_workbook
from pyvis.network import Network


_cwd = Path.cwd().resolve()
_project_candidate = None
for _candidate in (_cwd, *_cwd.parents):
    if (_candidate / "src").exists():
        _project_candidate = _candidate
        break
if _project_candidate is not None:
    _src_path = _project_candidate / "src"
    if str(_src_path) not in sys.path:
        sys.path.insert(0, str(_src_path))

from chec_impacto.notebook_support import find_repo_root

REPO_ROOT = find_repo_root()
JSON_PATH = REPO_ROOT / 'src/data/variables.json'
FULL_OUTPUT = REPO_ROOT / 'src/assets/site/results/grafo_red_nivel2.html'
SELECTED_OUTPUT = REPO_ROOT / 'src/assets/site/results/grafo_red_nivel2_seleccionadas.html'
MGCECDL_OUTPUT = REPO_ROOT / 'src/assets/site/results/grafo_red_nivel2_mgcecdl.html'
MGCECDL_PROJECT_ROOT = REPO_ROOT

MODE_COLORS = {
    'A': '#e74c3c',
    'B': '#f39c12',
    'C': '#9b59b6',
    'D': '#3498db',
    'E': '#1abc9c',
    'F': '#2ecc71',
}

In [3]:
from chec_impacto.data.graph_visualization import (
    build_edges,
    create_graph,
    expand_variables,
    load_selected_names,
    resolve_selection_path,
    save_graph,
    selected_variables,
)


In [4]:
# Graph edge construction is imported from `chec_impacto.data.graph_visualization`.


In [5]:
# Graph creation, preserved-edge logic, and HTML export are imported from `chec_impacto.data.graph_visualization`.


In [6]:
with JSON_PATH.open(encoding='utf-8') as file:
    config = json.load(file)

selection_path = resolve_selection_path(MGCECDL_PROJECT_ROOT, REPO_ROOT)
selected_names = load_selected_names(selection_path)
variables_by_mode, climate_families = expand_variables(config)
filtered_by_mode = selected_variables(
    variables_by_mode, selected_names, climate_families
)
mode_names = {mode['id']: mode['nombre'] for mode in config['modos']}
edges = build_edges(config)

full_graph = create_graph(variables_by_mode, edges, mode_names)
filtered_graph = create_graph(
    filtered_by_mode,
    edges,
    mode_names,
    preserve_filtered_connections=True,
)

assert full_graph.number_of_nodes() == config['totalVariables']
expected_selected_nodes = sum(len(variables) for variables in filtered_by_mode.values())
assert filtered_graph.number_of_nodes() == expected_selected_nodes

save_graph(
    full_graph,
    FULL_OUTPUT,
    variables_by_mode,
    mode_names,
    'Diccionario de modos - Grafo completo',
)
save_graph(
    filtered_graph,
    SELECTED_OUTPUT,
    filtered_by_mode,
    mode_names,
    'Diccionario de modos - Variables seleccionadas',
)

print(f'Excel de selección: {selection_path}')
print(f'Variables marcadas en el Excel: {len(selected_names)}')
print(
    f'Grafo completo: {full_graph.number_of_nodes()} nodos, '
    f'{full_graph.number_of_edges()} conexiones -> {FULL_OUTPUT}'
)
print(
    f'Grafo seleccionado: {filtered_graph.number_of_nodes()} nodos, '
    f'{filtered_graph.number_of_edges()} conexiones '
    f'({sum(1 for _, _, data in filtered_graph.edges(data=True) if data.get("dashes"))} preservadas por filtro) '
    f'-> {SELECTED_OUTPUT}'
)

# Grafo y matriz usados por MGCECDL, alineados con el preprocesamiento real.
src_path = MGCECDL_PROJECT_ROOT / 'src'
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))
for module_name in list(sys.modules):
    if module_name == 'chec_impacto' or module_name.startswith('chec_impacto.'):
        del sys.modules[module_name]

from chec_impacto.data import (
    construir_matriz_adyacencia_mgcecdl,
    procesar_dataset_completo,
)

datos_procesados = procesar_dataset_completo(
    path_clima=MGCECDL_PROJECT_ROOT / 'data/Indicadores_vano_v3.csv',
    path_variables_seleccion=selection_path,
    use_sampling=False,
    min_samples_per_codigo=5,
    target='UITI_VANO',
    filtro_uiti_max=None,
    ventana_climatica_horas=config['ventanaClimaticaHoras'],
)
features = datos_procesados['features']
feature_set = set(features)
mgcecdl_by_mode = {
    mode_id: [variable for variable in variables if variable in feature_set]
    for mode_id, variables in variables_by_mode.items()
}
mgcecdl_graph = create_graph(
    mgcecdl_by_mode,
    edges,
    mode_names,
    preserve_filtered_connections=True,
)
save_graph(
    mgcecdl_graph,
    MGCECDL_OUTPUT,
    mgcecdl_by_mode,
    mode_names,
    'Grafo MGCECDL - Orden del preprocesamiento',
)

adjacency_matrix, preserved_edges = construir_matriz_adyacencia_mgcecdl(
    features,
    ventana_climatica_horas=config['ventanaClimaticaHoras'],
)
graph_dir = MGCECDL_PROJECT_ROOT / 'data/graphs'
graph_dir.mkdir(parents=True, exist_ok=True)
np.save(graph_dir / 'mgcecdl_adjacency_matrix.npy', adjacency_matrix)
(graph_dir / 'mgcecdl_feature_order.json').write_text(
    json.dumps(features, ensure_ascii=False, indent=2),
    encoding='utf-8',
)
(graph_dir / 'mgcecdl_preserved_edges.json').write_text(
    json.dumps(preserved_edges, ensure_ascii=False, indent=2),
    encoding='utf-8',
)

assert adjacency_matrix.shape == (len(features), len(features))
assert set(mgcecdl_graph.nodes) == feature_set
assert 'UITI_VANO' not in feature_set
print(
    f'Grafo MGCECDL: {mgcecdl_graph.number_of_nodes()} nodos, '
    f'{mgcecdl_graph.number_of_edges()} conexiones -> {MGCECDL_OUTPUT}'
)
print(
    f'Matriz MGCECDL: {adjacency_matrix.shape}, '
    f'{np.count_nonzero(adjacency_matrix)} aristas -> {graph_dir}'
)

Excel de selección: /Users/diego/Desktop/Reporte/chec-local-uiti-vano-interpreter/data/Variables_seleccion.xlsx
Variables marcadas en el Excel: 27
Grafo completo: 156 nodos, 156 conexiones -> /Users/diego/Desktop/Reporte/chec-local-uiti-vano-interpreter/src/assets/site/results/grafo_red_nivel2.html
Grafo seleccionado: 71 nodos, 68 conexiones (16 preservadas por filtro) -> /Users/diego/Desktop/Reporte/chec-local-uiti-vano-interpreter/src/assets/site/results/grafo_red_nivel2_seleccionadas.html
Cargando datos...
Procesamiento completado.
Shape X: (159470, 70), Shape y: (159470, 1)
Grafo MGCECDL: 70 nodos, 56 conexiones -> /Users/diego/Desktop/Reporte/chec-local-uiti-vano-interpreter/src/assets/site/results/grafo_red_nivel2_mgcecdl.html
Matriz MGCECDL: (70, 70), 56 aristas -> /Users/diego/Desktop/Reporte/chec-local-uiti-vano-interpreter/data/graphs
